In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

dbutils.widgets.text("extractor_notebook_path", "")
dbutils.widgets.text("config_path", "")

extractor_notebook_path = dbutils.widgets.get("extractor_notebook_path")
config_path = dbutils.widgets.get("config_path")

with open(config_path) as f:
    triggers = json.load(f)

In [ ]:
def run_worker(trigger):
    return dbutils.notebook.run(
        "./worker",
        600,
        {
            "extractor_notebook_path": extractor_notebook_path,
            "trigger": json.dumps(trigger),
        },
    )

with ThreadPoolExecutor(max_workers=min(len(triggers), 8)) as executor:
    futures = [executor.submit(run_worker, trigger) for trigger in triggers]
    results = [json.loads(future.result()) for future in as_completed(futures)]

In [ ]:
with open(config_path, "w") as f:
    json.dump(results, f, indent=2)